# DeepFake-AI: Google Colab Pro & Drive Live Streaming Pipeline
**Authors**: Dar Rehan Rasool (Research Contributor), Dr. Riyaz (Supervisor), Dr. Javad (Supervisor)
**Affiliation**: Islamic University of Science and Technology (IUST), Kashmir

This notebook provides end-to-end execution for training and evaluating DeepFake-AI models on Google Colab Pro (A100 / V100 / T4) while **streaming 500GB Kaggle datasets directly through Google Drive**, preventing local laptop disk exhaustion.


In [ ]:
# 1. GPU Acceleration & System Telemetry Audit
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# 2. Mount Google Drive to bypass local storage
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path
DRIVE_DIR = Path("/content/drive/MyDrive/DeepFake_Research")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "data/raw").mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "checkpoints/best").mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "visualizations").mkdir(parents=True, exist_ok=True)
print(f"Project Workspace on Google Drive: {DRIVE_DIR}")


In [ ]:
# 3. Setup Kaggle API and Stream Competition Data to Drive
import shutil
kaggle_token_src = Path("/content/drive/MyDrive/kaggle.json")
kaggle_token_dst = Path.home() / ".kaggle/kaggle.json"

if kaggle_token_src.exists():
    kaggle_token_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(kaggle_token_src, kaggle_token_dst)
    os.chmod(kaggle_token_dst, 0o600)
    print("Kaggle credentials configured!")
    # Direct download to Google Drive target (not Colab local disk)
    # !kaggle competitions download -c deepfake-detection-challenge -p /content/drive/MyDrive/DeepFake_Research/data/raw/
else:
    print("Place kaggle.json in your Google Drive root (MyDrive/kaggle.json) to enable automatic downloads.")


In [ ]:
# 4. Clone / Link Repository Code in Colab
%cd /content
!git clone https://github.com/darrehanrasool/Deepfake.git deepfake_repo
%cd /content/deepfake_repo
!pip install -r requirements.txt


In [ ]:
# 5. Launch Training with Automatic Device Selection & Drive Checkpointing
import sys
sys.path.insert(0, "src")

!python3 scripts/colab_drive_runner.py --model st2_syncnet --epochs 10 --batch-size 32


In [ ]:
# 6. Generate 10 Publication-Grade Visualizations & Copy to Google Drive
!python3 scripts/generate_visualizations.py
!cp -r visualizations/* /content/drive/MyDrive/DeepFake_Research/visualizations/
print("All 10 visualizations synced to Google Drive!")
